# Semantic Business Layer for ManagedService-X

Builds a curated **business-entity (gold) layer** over `ManagedServiceData` so AI agents
(Fabric IQ / Foundry IQ) reason over **business objects and relationships** instead of raw tables.

**Entities produced (prefix `sem_`):**

| Table | Grain | Represents |
|-------|-------|------------|
| `sem_dim_customer` | one row per customer/tenant | Customer/Tenant (Inforcer posture + CRM account) |
| `sem_fact_assessment` | one assessment run | Assessment (Security, Copilot Readiness, Copilot Assessment) |
| `sem_fact_check` | one check result | Check / control finding |
| `sem_fact_posture` | one row per customer | Security posture snapshot + exposure |
| `sem_dim_recommendation` | catalog row | Recommendation catalog (SKU talking points) |
| `sem_fact_recommendation` | customer × rule | Managed-service opportunity (failed checks → recommended SKU) |

**Relationships:** `Customer 1—* Assessment`, `Assessment 1—* Check`, `Customer 1—1 Posture`,
`Customer 1—* Recommendation-opportunity`, `Recommendation-opportunity *—1 Recommendation-catalog`.

Join key across facts = `customer_key` (normalized tenant name); checks/assessments also share `assessment_id`.


In [ ]:
# Semantic Business Layer — setup & helpers
from pyspark.sql import functions as F, Window

# Default lakehouse: ManagedServiceData (schema-enabled, schema = dbo)
SCHEMA = "dbo"

def rd(table_name):
    """Read a source/gold Delta table from the default lakehouse (dbo schema)."""
    return spark.table(f"{SCHEMA}.{table_name}")

def write_gold(df, table_name):
    """Overwrite a curated semantic-layer table in the dbo schema."""
    (df.write.mode("overwrite")
       .option("overwriteSchema", "true")
       .format("delta")
       .saveAsTable(f"{SCHEMA}.{table_name}"))
    print(f"wrote {SCHEMA}.{table_name}: {df.count():,} rows, {len(df.columns)} cols")

def norm_key(col):
    """Normalize a tenant/customer name into a stable join key."""
    return F.lower(F.trim(col))

def to_num(col):
    """Best-effort numeric cast (strips % and other non-numeric characters)."""
    return F.regexp_replace(col.cast("string"), r"[^0-9.\-]", "").cast("double")

print("Helpers ready. Default lakehouse = ManagedServiceData (dbo).")


StatementMeta(, 3df726fe-a135-452f-ad0a-c1215cd5416a, 7, Finished, Available, Finished, False)

Helpers ready. Default lakehouse = ManagedServiceData (dbo).


In [ ]:
# Entity: Customer  ->  sem_dim_customer
# One row per customer/tenant, enriched with Inforcer posture and CRM account.

inf = rd("inforcer_tenants").select(
    norm_key("tenantFriendlyName").alias("customer_key"),
    F.col("tenantFriendlyName").alias("inf_name"),
    "clientTenantId", "msTenantId", "tenantDnsName",
    F.col("tenant_type").alias("inf_tenant_type"),
    # NOTE: This is the Inforcer backup/management plan tier (e.g. Premium/Standard),
    # NOT the customer's Microsoft 365 / Copilot licensing (not available in source).
    F.col("licenseSkus").alias("inforcer_plan_tier"),
    to_num(F.col("secureScore")).alias("secure_score"),
    "isBaseline", "lastBackupTimestamp",
)

corr = rd("inforcer_crm_correlation").select(
    "clientTenantId",
    F.col("AccountID").alias("corr_account_id"),
    F.col("CrmAccountName").alias("corr_account_name"),
    F.col("Owner").alias("corr_owner"),
    F.col("OwnerEmail").alias("corr_owner_email"),
    F.col("PrimaryContactEmail").alias("corr_contact_email"),
)

# CRM fallback via the domain-derived customer map (one row per tenant, prefer a CRM match)
w_ctd = Window.partitionBy(norm_key(F.col("TenantName"))).orderBy(
    F.when(F.lower(F.col("CrmMatch").cast("string")).isin("true", "1", "yes"), 0).otherwise(1),
    F.col("BuiltAt").desc_nulls_last(),
)
ctd = (rd("customer_tenant_domains")
    .withColumn("customer_key", norm_key(F.col("TenantName")))
    .withColumn("_rn", F.row_number().over(w_ctd))
    .filter(F.col("_rn") == 1)
    .select(
        "customer_key",
        F.col("TenantName").alias("ctd_name"),
        F.col("TenantDomain").alias("ctd_dns"),
        F.col("CrmAccountID").alias("ctd_account_id"),
        F.col("CrmAccountName").alias("ctd_account_name"),
        F.col("RelationshipType").alias("relationship_type"),
        F.col("CrmPrimaryContactEmail").alias("ctd_contact_email"),
        F.col("Seller").alias("ctd_owner"),
        F.col("SellerEmail").alias("ctd_owner_email"),
        F.col("Territory").alias("territory"),
        F.col("AccountStatus").alias("account_status"),
    ))

# Canonical tenant universe: all assessment families + Inforcer + CRM-domain map
def names_from(table):
    return rd(table).select(
        norm_key(F.col("tenant_name")).alias("customer_key"),
        F.col("tenant_name").alias("src_name"),
    )

base = (
    names_from("security_assessment_assessments")
    .unionByName(names_from("copilot_readiness_assessments"))
    .unionByName(names_from("copilot_assessment_assessments"))
    .unionByName(inf.select("customer_key", F.col("inf_name").alias("src_name")))
    .unionByName(ctd.select("customer_key", F.col("ctd_name").alias("src_name")))
    .filter(F.col("customer_key").isNotNull() & (F.col("customer_key") != ""))
)
w_name = Window.partitionBy("customer_key").orderBy(F.length("src_name").desc())
base = (base.withColumn("_rn", F.row_number().over(w_name))
             .filter(F.col("_rn") == 1)
             .select("customer_key", F.col("src_name").alias("tenant_name")))

sem_dim_customer = (
    base
    .join(inf, "customer_key", "left")
    .join(corr, "clientTenantId", "left")
    .join(ctd, "customer_key", "left")
    .select(
        "customer_key", "tenant_name", "clientTenantId", "msTenantId",
        F.coalesce("tenantDnsName", "ctd_dns").alias("tenant_dns"),
        F.coalesce("inf_tenant_type", F.lit("Assessment")).alias("tenant_type"),
        "inforcer_plan_tier", "secure_score", "isBaseline", "lastBackupTimestamp",
        F.coalesce("corr_account_id", "ctd_account_id").alias("crm_account_id"),
        F.coalesce("corr_account_name", "ctd_account_name").alias("crm_account_name"),
        "relationship_type",
        F.coalesce("corr_owner", "ctd_owner").alias("owner"),
        F.coalesce("corr_owner_email", "ctd_owner_email").alias("owner_email"),
        F.coalesce("corr_contact_email", "ctd_contact_email").alias("primary_contact_email"),
        "territory", "account_status",
    )
    .withColumn("has_crm_match", F.col("crm_account_id").isNotNull())
    .withColumn("built_at", F.current_timestamp())
    .dropDuplicates(["customer_key"])
)

write_gold(sem_dim_customer, "sem_dim_customer")
display(sem_dim_customer.limit(20))


StatementMeta(, 3df726fe-a135-452f-ad0a-c1215cd5416a, 8, Finished, Available, Finished, False)

wrote dbo.sem_dim_customer: 108 rows, 20 cols


SynapseWidget(Synapse.DataFrame, c557bc5c-681c-4695-987b-bcffe4db1354)

In [3]:
# Fact: Assessment  ->  sem_fact_assessment
# One row per assessment run across the three assessment families.

common = ["assessment_id", "tenant_name", "assessment_type", "assessment_name",
          "assessment_date", "assessment_time", "overall_score_pct",
          "passed_count", "failed_count", "warnings_count"]

def asmt(table, family):
    return rd(table).select(*common).withColumn("assessment_family", F.lit(family))

sem_fact_assessment = (
    asmt("security_assessment_assessments", "Security")
    .unionByName(asmt("copilot_readiness_assessments", "CopilotReadiness"))
    .unionByName(asmt("copilot_assessment_assessments", "CopilotAssessment"))
    .withColumn("customer_key", norm_key(F.col("tenant_name")))
    .withColumn("overall_score_pct", to_num(F.col("overall_score_pct")))
    .withColumn("assessment_date_parsed", F.to_date(F.col("assessment_date")))
)

write_gold(sem_fact_assessment, "sem_fact_assessment")
display(sem_fact_assessment.groupBy("assessment_family").count().orderBy("assessment_family"))


StatementMeta(, 3df726fe-a135-452f-ad0a-c1215cd5416a, 10, Finished, Available, Finished, False)

wrote dbo.sem_fact_assessment: 373 rows, 13 cols


SynapseWidget(Synapse.DataFrame, 1a7b7a1e-a738-4fd4-a451-4b711a716a14)

In [4]:
# Fact: Check  ->  sem_fact_check
# One row per control/check result (Security + Copilot Readiness share this schema).

chk_cols = ["check_id", "assessment_id", "tenant_name", "category", "subcategory",
            "category_label", "check_name", "business_rationale", "status",
            "priority", "frameworks", "framework_control", "framework_level"]

def chk(table, family):
    return rd(table).select(*chk_cols).withColumn("check_family", F.lit(family))

sem_fact_check = (
    chk("security_assessment_checks", "Security")
    .unionByName(chk("copilot_readiness_checks", "CopilotReadiness"))
    .withColumn("customer_key", norm_key(F.col("tenant_name")))
    .withColumn("is_issue", F.lower(F.coalesce(F.col("status"), F.lit(""))).rlike("fail|warn"))
)

write_gold(sem_fact_check, "sem_fact_check")
display(sem_fact_check.groupBy("check_family", "status").count().orderBy("check_family", "status"))


StatementMeta(, 3df726fe-a135-452f-ad0a-c1215cd5416a, 12, Finished, Available, Finished, False)

wrote dbo.sem_fact_check: 20,628 rows, 16 cols


SynapseWidget(Synapse.DataFrame, a78ded30-b069-4de6-850b-5888b39c8a23)

In [ ]:
# Fact: Security Posture  ->  sem_fact_posture
# One snapshot row per customer: secure score + latest score per assessment family.

def latest_score(table, out_col):
    w = Window.partitionBy(norm_key(F.col("tenant_name"))).orderBy(
        F.to_date(F.col("assessment_date")).desc_nulls_last(),
        F.col("assessment_time").desc_nulls_last(),
    )
    return (rd(table)
        .withColumn("customer_key", norm_key(F.col("tenant_name")))
        .withColumn("_rn", F.row_number().over(w))
        .filter(F.col("_rn") == 1)
        .select("customer_key", to_num(F.col("overall_score_pct")).alias(out_col)))

sec = latest_score("security_assessment_assessments", "latest_security_score")
rdy = latest_score("copilot_readiness_assessments", "latest_readiness_score")
cop = latest_score("copilot_assessment_assessments", "latest_copilot_score")

sem_fact_posture = (
    rd("sem_dim_customer").select("customer_key", "secure_score")
    .join(sec, "customer_key", "left")
    .join(rdy, "customer_key", "left")
    .join(cop, "customer_key", "left")
    .withColumn("security_exposure", F.lit(100.0) - F.col("secure_score"))
    # Copilot readiness is low across the board (0-71, avg ~27), and no M365/Copilot
    # license data exists. So "adoption opportunity" is reframed as an ENABLEMENT gap:
    # the further below readiness, the greater the managed-service enablement opportunity.
    .withColumn("readiness_gap", F.lit(100.0) - F.col("latest_readiness_score"))
    .withColumn(
        "enablement_opportunity",
        F.when(F.col("latest_readiness_score").isNull(), F.lit("Unknown"))
         .when(F.col("latest_readiness_score") < 30, F.lit("High"))
         .when(F.col("latest_readiness_score") < 50, F.lit("Medium"))
         .otherwise(F.lit("Low")),
    )
    .withColumn("built_at", F.current_timestamp())
)

write_gold(sem_fact_posture, "sem_fact_posture")
display(sem_fact_posture.limit(20))


StatementMeta(, 3df726fe-a135-452f-ad0a-c1215cd5416a, 14, Finished, Available, Finished, False)

wrote dbo.sem_fact_posture: 108 rows, 7 cols


SynapseWidget(Synapse.DataFrame, 156a2b0a-ef75-4aec-9514-25dd3e214c49)

In [6]:
# Recommendation catalog + tenant opportunities
#   sem_dim_recommendation : the catalog (talking points / SKUs)
#   sem_fact_recommendation: managed-service opportunities = failed checks matched to catalog

sem_dim_recommendation = rd("recommendation_catalog")
write_gold(sem_dim_recommendation, "sem_dim_recommendation")

cat = (rd("recommendation_catalog")
    .filter(F.lower(F.coalesce(F.col("active").cast("string"), F.lit("true"))).isin("true", "1", "yes"))
    .select("rule_id", "assessment_type", "match_category", "match_keyword",
            "recommendation_sku", "recommendation_type", "talking_point",
            to_num(F.col("impact_weight")).alias("impact_weight")))

issues = (rd("sem_fact_check").filter(F.col("is_issue"))
          .select("customer_key", "check_family", "category", "subcategory", "check_name"))

# Match when the catalog keyword appears in the check category / subcategory / name
matched = issues.crossJoin(F.broadcast(cat)).where(
    (F.col("match_keyword").isNotNull()) & (F.col("match_keyword") != "") & (
        F.lower(F.coalesce(F.col("category"), F.lit(""))).contains(F.lower(F.col("match_keyword"))) |
        F.lower(F.coalesce(F.col("subcategory"), F.lit(""))).contains(F.lower(F.col("match_keyword"))) |
        F.lower(F.coalesce(F.col("check_name"), F.lit(""))).contains(F.lower(F.col("match_keyword")))
    )
)

sem_fact_recommendation = (
    matched.groupBy("customer_key", "rule_id", "recommendation_sku",
                    "recommendation_type", "talking_point", "impact_weight", "assessment_type")
    .agg(F.count(F.lit(1)).alias("matched_issue_count"))
    .withColumn("built_at", F.current_timestamp())
)

write_gold(sem_fact_recommendation, "sem_fact_recommendation")
display(sem_fact_recommendation.orderBy(F.col("matched_issue_count").desc()).limit(20))


StatementMeta(, 3df726fe-a135-452f-ad0a-c1215cd5416a, 16, Finished, Available, Finished, False)

wrote dbo.sem_dim_recommendation: 49 rows, 11 cols
wrote dbo.sem_fact_recommendation: 1,589 rows, 9 cols


SynapseWidget(Synapse.DataFrame, 631675ff-d517-404e-ad2e-f17c631b4bdd)

In [7]:
# Validation: row counts + relationship integrity
for t in ["sem_dim_customer", "sem_fact_assessment", "sem_fact_check",
          "sem_fact_posture", "sem_dim_recommendation", "sem_fact_recommendation"]:
    print(f"{t:26s} -> {rd(t).count():>7,} rows")

cust_keys = rd("sem_dim_customer").select("customer_key")
print("\nCustomers with a CRM match :", rd("sem_dim_customer").filter(F.col("has_crm_match")).count())
print("Assessments w/o a customer :", rd("sem_fact_assessment").join(cust_keys, "customer_key", "left_anti").count())
print("Checks w/o a customer      :", rd("sem_fact_check").join(cust_keys, "customer_key", "left_anti").count())
print("Opportunities (rows)       :", rd("sem_fact_recommendation").count())


StatementMeta(, 3df726fe-a135-452f-ad0a-c1215cd5416a, 18, Finished, Available, Finished, False)

sem_dim_customer           ->     108 rows
sem_fact_assessment        ->     373 rows
sem_fact_check             ->  20,628 rows
sem_fact_posture           ->     108 rows
sem_dim_recommendation     ->      49 rows
sem_fact_recommendation    ->   1,589 rows

Customers with a CRM match : 4
Assessments w/o a customer : 0
Checks w/o a customer      : 0
Opportunities (rows)       : 1589


## Reference — Semantic model relationships & DAX measures

Apply these when building the Power BI semantic model over the `sem_*` tables
(and generate the Fabric IQ ontology from that model).

### Relationships (single-direction, one-to-many unless noted)

| From (one) | Column | To (many) | Column | Cardinality |
|------------|--------|-----------|--------|-------------|
| `sem_dim_customer` | `customer_key` | `sem_fact_assessment` | `customer_key` | 1 → * |
| `sem_dim_customer` | `customer_key` | `sem_fact_check` | `customer_key` | 1 → * |
| `sem_dim_customer` | `customer_key` | `sem_fact_posture` | `customer_key` | 1 → 1 |
| `sem_dim_customer` | `customer_key` | `sem_fact_recommendation` | `customer_key` | 1 → * |
| `sem_fact_assessment` | `assessment_id` | `sem_fact_check` | `assessment_id` | 1 → * |
| `sem_dim_recommendation` | `rule_id` | `sem_fact_recommendation` | `rule_id` | 1 → * |

`sem_dim_customer` and `sem_dim_recommendation` are dimensions; the `sem_fact_*` tables are facts.

### DAX measures (create on the fact tables)

```dax
-- Posture
Copilot Readiness =
AVERAGE ( sem_fact_posture[latest_readiness_score] )

Security Score =
AVERAGE ( sem_fact_posture[secure_score] )

Security Exposure =
AVERAGE ( sem_fact_posture[security_exposure] )   -- = 100 - secure_score

Copilot Assessment Score =
AVERAGE ( sem_fact_posture[latest_copilot_score] )

-- Assessment volume
Assessment Count =
DISTINCTCOUNT ( sem_fact_assessment[assessment_id] )

Avg Overall Score =
AVERAGE ( sem_fact_assessment[overall_score_pct] )

-- Checks
Open Issues =
CALCULATE ( COUNTROWS ( sem_fact_check ), sem_fact_check[is_issue] = TRUE () )

Failed Check Rate =
DIVIDE ( [Open Issues], COUNTROWS ( sem_fact_check ) )

-- Opportunities
Open Recommendations =
DISTINCTCOUNT ( sem_fact_recommendation[rule_id] )

Opportunity Weight =
SUMX (
    sem_fact_recommendation,
    sem_fact_recommendation[impact_weight] * sem_fact_recommendation[matched_issue_count]
)

-- Customer coverage
Customers =
DISTINCTCOUNT ( sem_dim_customer[customer_key] )

CRM Matched Customers =
CALCULATE ( [Customers], sem_dim_customer[has_crm_match] = TRUE () )

-- Business flag: high-value Copilot adoption opportunity
Adoption Opportunity =
VAR Readiness = [Copilot Readiness]
VAR HasCopilot =
    CONTAINSSTRING (
        CALCULATE ( CONCATENATEX ( sem_dim_customer, sem_dim_customer[licenseSkus], "|" ) ),
        "COPILOT"
    )
RETURN
    IF ( Readiness > 80 && NOT HasCopilot, "High", "Low" )
```


In [9]:
# Inspect gold-table schemas — confirm exact column names for the DAX measures
sem_tables = [
    "sem_dim_customer",
    "sem_dim_recommendation",
    "sem_fact_assessment",
    "sem_fact_check",
    "sem_fact_posture",
    "sem_fact_recommendation",
]

for t in sem_tables:
    cols = spark.table(f"dbo.{t}").columns
    print(f"{t} ({len(cols)} cols):")
    for c in cols:
        print(f"    {c}")
    print()


StatementMeta(, f98b157a-b62d-4dfb-9cd5-4c4b1ab899ab, 3, Finished, Available, Finished, False)

sem_dim_customer (20 cols):
    customer_key
    tenant_name
    clientTenantId
    msTenantId
    tenant_dns
    tenant_type
    licenseSkus
    secure_score
    isBaseline
    lastBackupTimestamp
    crm_account_id
    crm_account_name
    relationship_type
    owner
    owner_email
    primary_contact_email
    territory
    account_status
    has_crm_match
    built_at

sem_dim_recommendation (11 cols):
    rule_id
    assessment_type
    match_category
    match_keyword
    recommendation_sku
    recommendation_type
    talking_point
    impact_weight
    active
    created_at
    updated_at

sem_fact_assessment (13 cols):
    assessment_id
    tenant_name
    assessment_type
    assessment_name
    assessment_date
    assessment_time
    overall_score_pct
    passed_count
    failed_count
    warnings_count
    assessment_family
    customer_key
    assessment_date_parsed

sem_fact_check (16 cols):
    check_id
    assessment_id
    tenant_name
    category
    subcategory
    ca

In [ ]:
# Ground-truth check: high Copilot readiness + no Copilot license
from pyspark.sql import functions as F

cust = spark.table("dbo.sem_dim_customer").select("customer_key", "tenant_name", "licenseSkus")
post = spark.table("dbo.sem_fact_posture").select("customer_key", "latest_readiness_score")

j = cust.join(post, "customer_key", "left")

# 1) How is licenseSkus actually stored? (format matters for the "no Copilot" filter)
print("=== Sample licenseSkus values ===")
j.select("licenseSkus").where(F.col("licenseSkus").isNotNull()).distinct().show(20, truncate=False)

# 2) Readiness distribution
print("=== Readiness score coverage ===")
j.select(
    F.count("*").alias("total"),
    F.count("latest_readiness_score").alias("has_readiness"),
    F.sum(F.when(F.col("latest_readiness_score") >= 80, 1).otherwise(0)).alias("readiness_ge_80"),
).show()

# 3) High readiness AND licenseSkus does not mention copilot
hi = j.filter(F.col("latest_readiness_score") >= 80)
no_cop = hi.filter(~F.lower(F.coalesce(F.col("licenseSkus"), F.lit(""))).contains("copilot"))
print(f"High readiness (>=80): {hi.count()}   |   of those WITHOUT 'copilot' in licenseSkus: {no_cop.count()}")

no_cop.select("tenant_name", "latest_readiness_score", "licenseSkus").orderBy(
    F.col("latest_readiness_score").desc()
).show(20, truncate=False)


StatementMeta(, 9d0be590-1651-48b6-84a0-9a26b637763a, 6, Finished, Available, Finished, False)

=== Sample licenseSkus values ===
+-----------+
|licenseSkus|
+-----------+
|PREMIUM    |
+-----------+

=== Readiness score coverage ===
+-----+-------------+---------------+
|total|has_readiness|readiness_ge_80|
+-----+-------------+---------------+
|  108|           97|              0|
+-----+-------------+---------------+

High readiness (>=80): 0   |   of those WITHOUT 'copilot' in licenseSkus: 0
+-----------+----------------------+-----------+
|tenant_name|latest_readiness_score|licenseSkus|
+-----------+----------------------+-----------+
+-----------+----------------------+-----------+



In [2]:
# Root-cause diagnostics for readiness scale + license SKU source
from pyspark.sql import functions as F

# 1) Readiness score distribution (pick a correct threshold / verify scale)
post = spark.table("dbo.sem_fact_posture")
print("=== latest_readiness_score distribution ===")
post.select(
    F.min("latest_readiness_score").alias("min"),
    F.expr("percentile_approx(latest_readiness_score, 0.5)").alias("median"),
    F.avg("latest_readiness_score").alias("avg"),
    F.expr("percentile_approx(latest_readiness_score, 0.9)").alias("p90"),
    F.max("latest_readiness_score").alias("max"),
).show()

# 2) Where do Copilot licenses actually live? Inspect inforcer_tenant_licenses SKUs
lic = spark.table("dbo.inforcer_tenant_licenses")
print("=== inforcer_tenant_licenses columns ===", lic.columns)
print("=== distinct SKUs mentioning 'copilot' ===")
lic.select("sku").where(F.lower(F.col("sku")).contains("copilot")).distinct().show(50, truncate=False)
print("=== top SKUs overall ===")
lic.groupBy("sku").count().orderBy(F.col("count").desc()).show(30, truncate=False)

# 3) Confirm licenseSkus in the source tenants table
ten = spark.table("dbo.inforcer_tenants")
print("=== inforcer_tenants.licenseSkus sample ===")
ten.select("tenantFriendlyName", "licenseSkus").show(10, truncate=False)


StatementMeta(, 9d0be590-1651-48b6-84a0-9a26b637763a, 7, Finished, Available, Finished, False)

=== latest_readiness_score distribution ===
+---+------+-----------------+----+----+
|min|median|              avg| p90| max|
+---+------+-----------------+----+----+
|0.0|  29.0|26.95876288659794|43.0|71.0|
+---+------+-----------------+----+----+

=== inforcer_tenant_licenses columns === ['parent_id', 'sku']
=== distinct SKUs mentioning 'copilot' ===
+---+
|sku|
+---+
+---+

=== top SKUs overall ===
+-------+-----+
|sku    |count|
+-------+-----+
|PREMIUM|15   |
+-------+-----+

=== inforcer_tenants.licenseSkus sample ===
+------------------------------------+-----------+
|tenantFriendlyName                  |licenseSkus|
+------------------------------------+-----------+
|Reliance Software Design            |PREMIUM    |
|KINETIC TOURS COMPANY LIMITED       |PREMIUM    |
|DANG Lifestyle Inc                  |PREMIUM    |
|Ghana College of Nurses and Midwives|PREMIUM    |
|University of Gold Coast            |PREMIUM    |
|Ebony Oil & Gas Limited             |PREMIUM    |
|RG Estate 